<a href="https://colab.research.google.com/github/Nitinprasadsingh/Synapse/blob/main/baselineXGboostmodel(NSRDB)(final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("/content/df_with_pv_data (2).csv")


# ============================================================
# 2. CREATE DATETIME AND SORT CHRONOLOGICALLY
# ============================================================

df["Datetime"] = pd.to_datetime(
    df[["Year", "Month", "Day", "Hour", "Minute"]]
)

df = df.sort_values("Datetime").reset_index(drop=True)


# ============================================================
# 3. FEATURES
#    BASELINE 2 = WITHOUT POA_IRRADIANCE
# ============================================================

features = [
    "Temperature",
    "Wind Speed",
    "Wind Direction",
    "Pressure",
    "Solar Zenith Angle",
    "DNI",
    "GHI",
    "DHI",
    "Relative Humidity",
    "Dew Point",
    "Hour_sin",
    "Hour_cos",
    "DayOfYear"
]

target = "PV_target"


# ============================================================
# 4. REMOVE MISSING VALUES
# ============================================================

df_model = df[features + [target]].dropna().reset_index(drop=True)

X = df_model[features]
y = df_model[target]


# ============================================================
# 5. CHRONOLOGICAL 80/20 TRAIN-TEST SPLIT
# ============================================================

split_index = int(len(df_model) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]


print("==============================")
print("DATA SPLIT")
print("==============================")

print(f"Total samples    : {len(df_model)}")
print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")


# ============================================================
# 6. TRAIN XGBOOST
# ============================================================

model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


# ============================================================
# 7. PREDICTION
# ============================================================

y_pred = model.predict(X_test)


# ============================================================
# 8. OVERALL TEST PERFORMANCE
# ============================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)


print("\n==============================")
print("BASELINE 2 - WITHOUT POA")
print("==============================")

print(f"MAE  : {mae:.4f} kW")
print(f"RMSE : {rmse:.4f} kW")
print(f"R²   : {r2:.4f}")


# ============================================================
# 9. DAYTIME PERFORMANCE
# ============================================================

daytime = y_test.values > 0

day_mae = mean_absolute_error(
    y_test.values[daytime],
    y_pred[daytime]
)

day_rmse = np.sqrt(
    mean_squared_error(
        y_test.values[daytime],
        y_pred[daytime]
    )
)

day_r2 = r2_score(
    y_test.values[daytime],
    y_pred[daytime]
)

# 100 kW installed capacity
nmae = (day_mae / 100) * 100


print("\n==============================")
print("DAYTIME PERFORMANCE")
print("==============================")

print(f"MAE  : {day_mae:.4f} kW")
print(f"RMSE : {day_rmse:.4f} kW")
print(f"R²   : {day_r2:.4f}")
print(f"nMAE : {nmae:.2f}%")


# ============================================================
# 10. PERMUTATION IMPORTANCE
# ============================================================

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

perm_importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
})

perm_importance = perm_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)


print("\n==============================")
print("PERMUTATION IMPORTANCE")
print("==============================")

print(
    perm_importance.to_string(index=False)
)


# ============================================================
# 11. SAVE PREDICTIONS
# ============================================================

results = pd.DataFrame({
    "Datetime": df.loc[X_test.index, "Datetime"].values,
    "Actual_PV_kW": y_test.values,
    "Predicted_PV_kW": y_pred
})

results["Error_kW"] = (
    results["Predicted_PV_kW"] -
    results["Actual_PV_kW"]
)

results["Absolute_Error_kW"] = (
    results["Error_kW"].abs()
)

results.to_csv(
    "baseline_2_predictions.csv",
    index=False
)

print("\nPredictions saved to:")
print("baseline_2_predictions.csv")

DATA SPLIT
Total samples    : 105104
Training samples : 84083
Testing samples  : 21021

BASELINE 2 - WITHOUT POA
MAE  : 2.3164 kW
RMSE : 5.4241 kW
R²   : 0.9403

DAYTIME PERFORMANCE
MAE  : 4.5128 kW
RMSE : 7.6423 kW
R²   : 0.8863
nMAE : 4.51%

PERMUTATION IMPORTANCE
           Feature  Importance
               GHI   10.976622
Solar Zenith Angle    2.390340
               DNI    2.249290
               DHI    1.669998
 Relative Humidity    0.601903
          Hour_sin    0.585329
       Temperature    0.382989
         DayOfYear    0.271458
          Pressure    0.236036
          Hour_cos    0.209075
         Dew Point    0.192105
        Wind Speed    0.157878
    Wind Direction    0.062924

Predictions saved to:
baseline_2_predictions.csv


PROJECT: AI-Driven Micro-Grid Load Balancing and Solar Energy Forecasting

BASELINE MODEL STATUS: FINALIZED

Objective:
Predict solar PV power generation 1 hour ahead (t+1) using historical
NSRDB weather, irradiance, solar geometry, and time features.

DATA:
- Dataset: NSRDB
- Year used for baseline: 2018
- Resolution: Hourly
- Total samples after preprocessing: 8,759
- Train/test split: chronological 80/20
- Training samples: 7,007
- Testing samples: 1,752
- NO random sampling

TARGET:
PV_target = PV_Output_kW at the next hour

Physics-based PV output generation:
1. Solar zenith angle converted to radians.
2. Panel tilt = 33°
3. Panel azimuth = 180° (south-facing)
4. Beam POA:
   DNI × cos(angle_of_incidence)
5. Diffuse POA:
   DHI × (1 + cos(panel_tilt)) / 2
6. POA clipped to >= 0
7. PV output:
   PV_Output_kW =
       100 kW × (POA_Irradiance / 1000) × 0.80

PV assumptions:
- Installed PV capacity: 100 kW
- System efficiency/derating: 80%
- Panel tilt: 33°
- Panel azimuth: 180°
- Surface albedo assumption: 0.2
- NOTE: Current simplified POA calculation does not yet include the
  ground-reflected/albedo component or full solar azimuth-based AOI.

FINAL BASELINE INPUT FEATURES:

[
    Temperature,
    Wind Speed,
    Wind Direction,
    Pressure,
    Solar Zenith Angle,
    DNI,
    GHI,
    DHI,
    Relative Humidity,
    Dew Point,
    Hour_sin,
    Hour_cos,
    DayOfYear
]

IMPORTANT:
- POA_Irradiance is NOT used as an ML input.
- PV_Output_kW is NOT used as an ML input.
- PV_target is the prediction target.
- POA_Irradiance and PV_Output_kW are used only for physics-based
  target generation.

MODEL:
- Algorithm: XGBoost Regressor
- n_estimators = 500
- max_depth = 6
- learning_rate = 0.05
- subsample = 0.8
- colsample_bytree = 0.8
- objective = reg:squarederror
- random_state = 42
- n_jobs = -1

FINAL BASELINE PERFORMANCE:

Daytime:
- MAE  = 3.5857 kW
- RMSE = 5.9308 kW
- R²   = 0.9377
- nMAE = 3.59% of 100 kW installed capacity

PERSISTENCE BASELINE:
- MAE  = 10.7273 kW
- RMSE = 12.9644 kW
- R²   = 0.7021
- nMAE = 10.73%

The XGBoost baseline reduces daytime MAE by approximately 66.6%
compared with persistence.

FEATURE IMPORTANCE FROM FINAL BASELINE:

1. GHI                 11.7324
2. Hour_sin             4.3908
3. DNI                   4.2849
4. Solar Zenith Angle   1.0360
5. DHI                   0.9376
6. Hour_cos              0.8092
7. Relative Humidity     0.2486
8. Wind Speed             0.0717
9. Wind Direction         0.0315
10. Pressure              0.0246
11. Dew Point             0.0185
12. Temperature           0.0043
13. DayOfYear             0.0000

FEATURE ABLATION RESULTS:

Experiment B — Removed time features:
- Daytime MAE  = 7.0234 kW
- Daytime RMSE = 10.2169 kW
- Daytime R²   = 0.8150
Conclusion:
Time features significantly contribute to forecasting performance.
Hour_sin and Hour_cos should remain in the baseline.

Experiment C — Added current PV + lagged irradiance:
- Added PV_current
- Added GHI_lag1
- Added DNI_lag1
- Added DHI_lag1
- Daytime MAE  = 3.4759 kW
- Daytime RMSE = 5.7953 kW
- Daytime R²   = 0.9405
Conclusion:
Slight improvement over the final baseline, but not enough to replace
the clean weather-only baseline. PV_current was the dominant added
feature.

Experiment D — Added current PV only:
- Daytime MAE  = 3.7073 kW
- Daytime RMSE = 5.9365 kW
- Daytime R²   = 0.9375

NOTE:
PV_Output_kW is mathematically proportional to POA_Irradiance:
PV_Output_kW = 0.08 × POA_Irradiance.
Therefore PV_current and POA contain essentially the same information
in this synthetic/physics-derived dataset.

FINAL DECISION:
The official baseline is the clean NSRDB → XGBoost model using:
Weather + irradiance + solar geometry + time features.

Baseline benchmark to beat:
Daytime MAE = 3.5857 kW
Daytime R²  = 0.9377

NEXT PROJECT STAGE:
Introduce satellite-derived cloud information/CNN features and test
whether they improve the 1-hour-ahead PV forecast beyond this baseline.

Future deployment consideration:
Training currently uses NSRDB historical irradiance values. During
real deployment, future weather/irradiance will need to come from
forecast sources such as Open-Meteo. Satellite cloud features will
potentially provide additional information about cloud evolution and
rapid PV changes.

PRIMARY RESEARCH/ENGINEERING QUESTION:
Can satellite-derived cloud information improve 1-hour-ahead PV
forecasting beyond the 3.59 kW daytime MAE achieved by the NSRDB
weather-based XGBoost baseline?
:::